<a href="https://colab.research.google.com/github/Shiveshrane/Research_paper_implementations/blob/main/Mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import math

In [ ]:
embed_dim=32
max_seq_len=64

In [ ]:
theta=tf.expand_dims(theta, axis=0)
theta.shape

TensorShape([1, 16])

In [ ]:
tf.sin(positions*theta), tf.cos(positions*theta)

(<tf.Tensor: shape=(64, 16), dtype=float32, numpy=
 array([[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 8.4147096e-01,  3.1098360e-01,  9.9833421e-02, ...,
          3.1622776e-07,  1.0000000e-07,  3.1622776e-08],
        [ 9.0929741e-01,  5.9112710e-01,  1.9866933e-01, ...,
          6.3245551e-07,  2.0000000e-07,  6.3245551e-08],
        ...,
        [-9.6611774e-01,  4.2624542e-01, -1.8216260e-01, ...,
          1.9289893e-05,  6.1000001e-06,  1.9289894e-06],
        [-7.3918068e-01,  6.8642765e-01, -8.3089121e-02, ...,
          1.9606121e-05,  6.1999999e-06,  1.9606121e-06],
        [ 1.6735570e-01,  8.7853855e-01,  1.6814092e-02, ...,
          1.9922349e-05,  6.3000002e-06,  1.9922350e-06]], dtype=float32)>,
 <tf.Tensor: shape=(64, 16), dtype=float32, numpy=
 array([[ 1.        ,  1.        ,  1.        , ...,  1.        ,
          1.        ,  1.        ],
        [ 0.5403023 ,  0.95041525,  0.9950042

# Rotary Embeddings

In [ ]:
class RotaryEmbeddings(tf.keras.layers.Layer):
  def __init__(self, embed_dim, max_seq_len):
    super().__init__()
    self.seq_len=max_seq_len
    self.theta=tf.pow(10000.0, -2*tf.range(0, embed_dim//2, dtype=tf.float32)/tf.cast(embed_dim, tf.float32))
    self.positions=tf.range(0, max_seq_len, dtype=tf.float32)
    self.positions=tf.reshape(self.positions, shape=(-1,1))
    self.cos=tf.cos(self.positions*self.theta)
    self.sin=tf.sin(self.positions*self.theta)

  def call(self, x, start_pos):
    batch_size, seq_len , n_heads, embed_dim=x.shape
    x_reshaped=tf.reshape(x, shape=(batch_size, seq_len, n_heads, embed_dim//2, 2))
    start_pos=tf.convert_to_tensor(start_pos, dtype=tf.int32)
    cos=tf.slice(self.cos, [start_pos, 0], [seq_len, embed_dim//2])
    cos=tf.expand_dims(tf.expand_dims(cos, axis=0), 2)
    sin=tf.slice(self.sin, [start_pos, 0], [seq_len, embed_dim//2])
    sin=tf.expand_dims(tf.expand_dims(sin, axis=0), 2)

    x_0=x_reshaped[..., 0]
    x_1=x_reshaped[..., 1]

    x_embedded=tf.stack([x_0*cos-x_1*sin, x_0*sin + x_1*cos], axis=1)
    x_embedded=tf.reshape(x_embedded, shape=(batch_size, seq_len, n_heads, embed_dim))
    return x_embedded




In [ ]:
X=tf.random.normal(shape=(1, 64, 8, 32))
embedding_dim=32
max_seq_len=64

embd=RotaryEmbeddings(embedding_dim, max_seq_len)(X, start_pos=0)

## RMS Norm

In [ ]:
class RMSNorm(tf.keras.layers.Layer):
  def __init__(self, dim, eps=1e-6):
    super().__init__()
    self.eps=eps
    self.weight=self.add_weight(
        shape=(dim,),
        name="weights",
        initializer="ones",
        trainable=True
    )
  def rms_calc(self, x):
    rms_value=tf.math.sqrt(tf.reduce_mean(tf.square(x), axis=-1, keepdims=True)+self.eps)
    return x/rms_value

  def call(self, x):
    return self.weight*self.rms_calc(x)

## Feed Forward Neural Network

In [ ]:
class FFN(tf.keras.layers.Layer):
  def __init__(self, dim, multiple_of=256, custom_mult=None, dropout=0.2):
    super().__init__()
    self.dim=dim
    self.hidden_dim=dim
    self.dropout=dropout
    if custom_mult is not None:
      self.hidden_dim=int(dim*custom_mult)

    self.w1=tf.keras.layers.Dense(self.dim, use_bias=False)
    self.w2=tf.keras.layers.Dense(self.dim, use_bias=False)
    self.w3=tf.keras.layers.Dense(self.hidden_dim, use_bias=False)
    self.dropout_layer=tf.keras.layers.Dropout(self.dropout)

  def call(self, x):
    # output=w3(silu(w1(x))+w2(x))
    x1=self.w1(x)
    x2=self.w2(x)
    x3=self.w3(tf.nn.silu(x1)+x2)
    x3=self.dropout_layer(x3)
    return x3



## Sliding Window Attention

In [ ]:

    def _create_sliding_window_mask( seq_len, window_size, causal=False):
        """
        Creates a sliding window attention mask.
        - If causal=True, each token attends to the previous window_size tokens.
        - If causal=False, each token attends to window_size tokens before and after.
        """
        # Create indices for rows (queries) and columns (keys)
        indices = tf.range(seq_len)
        row_indices = tf.expand_dims(indices, 1)  # [seq_len, 1]
        col_indices = tf.expand_dims(indices, 0)  # [1, seq_len]

        # Compute the distance between each query and key position
        distances = row_indices - col_indices  # [seq_len, seq_len]

        if causal:
            # Causal case: only attend to past tokens within window_size
            mask = tf.where(
                (distances >= 0) & (distances <= window_size),  # Within window
                0.0,  # Attend
                tf.float32.min  # Don't attend
            )
        else:
            # Non-causal case: attend to ±window_size around each token
            mask = tf.where(
                tf.abs(distances) <= window_size,  # Within window
                0.0,  # Attend
                tf.float32.min  # Don't attend
            )

        return mask

In [ ]:
_create_sliding_window_mask(5, 2, causal=True)

<tf.Tensor: shape=(5, 5), dtype=float32, numpy=
array([[ 0.0000000e+00, -3.4028235e+38, -3.4028235e+38, -3.4028235e+38,
        -3.4028235e+38],
       [ 0.0000000e+00,  0.0000000e+00, -3.4028235e+38, -3.4028235e+38,
        -3.4028235e+38],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, -3.4028235e+38,
        -3.4028235e+38],
       [-3.4028235e+38,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
        -3.4028235e+38],
       [-3.4028235e+38, -3.4028235e+38,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00]], dtype=float32)>

In [ ]:
class SWA(tf.keras.layers.Layer):
  def __init__(self, embed_dim, max_seq_len, n_heads, n_kv_heads, window_size, use_cache=False):
    super().__init__()

    self.dim=embed_dim
    self.seq_len=max_seq_len
    self.max_seq_len=max_seq_len
    self.n_heads=n_heads
    self.head_dim=embed_dim//n_heads
    self.n_kv_heads=n_kv_heads
    self.repeats=self.n_heads//self.n_kv_heads


    self.Wq=tf.keras.layers.Dense(self.n_heads*self.head_dim, use_bias=False)
    self.Wk=tf.keras.layers.Dense(self.n_kv_heads*self.head_dim, use_bias=False)
    self.Wv=tf.keras.layers.Dense(self.n_kv_heads*self.head_dim, use_bias=False)
    self.Wout=tf.keras.layers.Dense(self.n_heads*self.head_dim, use_bias=False)

    self.rotary_embeddings=RotaryEmbeddings(self.dim, self.max_seq_len)
    self.rms_norm=RMSNorm(self.dim)

    self.key_cache=self.add_weight(
        shape=(1, self.max_seq_len, self.n_kv_heads, self.head_dim),
        name="key_cache",
        initializer="zeros",
        trainable=False
    )

    self.value_cache=self.add_weight(
        shape=(1, self.max_seq_len, self.n_kv_heads, self.head_dim),
        name="value_cache",
        initializer="zeros",
        trainable=False
    )
    self.window_size=window_size


  #Adding sliding_window_attn logic
  def sliding_window_attn(self,seq_len, causal=True):
    indices=tf.range(seq_len)
    row_indices=tf.expand_dims(indices,1)
    col_indices=tf.expand_dims(indices, 0)
    distances=row_indices-col_indices

    if causal:
      mask=tf.where(
      (distances>=0) & (distances<=self.window_size),
      0.0,
      tf.float32.min
      )
    else:
      mask=tf.where(
          tf.abs(distances)<=self.window_size,
          0.0,
          tf.float32.min
      )

    return mask

  #Add a rolling buffer cache logic


  #Calling the attention mechanism
  def call(self, X, mask=None, causal=False):
    batch_size, seq_len, embed_dim=X.shape
    q=self.Wq(X)
    k=self.Wk(X)
    v=self.Wv(X)
    q=tf.reshape(q, shape=(batch_size, seq_len, self.n_heads, self.head_dim))
    k=tf.reshape(k, shape=(batch_size, seq_len, self.n_kv_heads, self.head_dim))
    v=tf.reshape(v, shape=(batch_size, seq_len, self.n_kv_heads, self.head_dim))
    q_rotary=self.rotary_embeddings(q, start_pos=0)
    k_rotary=self.rotary_embeddings(k, start_pos=0)


    ##Repeat k and v
    k=tf.repeat(k_rotary, self.repeats, axis=2)
    v=tf.repeat(v, self.repeats, axis=2)


    q=tf.transpose(q_rotary, perm=[0, 2, 1, 3])
    k=tf.transpose(k, perm=[0, 2, 1, 3])
    v=tf.transpose(v, perm=[0, 2, 1, 3])


    scores=tf.matmul(q, k, transpose_b=True)/tf.sqrt(tf.cast(self.head_dim, tf.float32))
    if causal:
      mask=self.sliding_window_attn(seq_len=seq_len,causal=causal)
      mask=tf.expand_dims(tf.expand_dims(mask, 0), 0)
     # print(mask)

      scores=scores+mask
    if mask is not None:
      scores=scores+mask

    attn=tf.nn.softmax(scores, axis=-1)
    out=tf.matmul(attn, v)

    out=tf.transpose(out, perm=[0, 2, 1, 3])
    out=tf.reshape(out, shape=(batch_size, seq_len, self.n_heads*self.head_dim))
    out= self.Wout(out)

    return out




In [ ]:
## Testing SWA

def test_sliding_window_attention():
    # Configuration parameters
    batch_size = 1
    seq_len = 16
    embed_dim = 64
    n_heads = 4
    n_kv_heads = 2
    window_size = 4

    # Create a simple input sequence
    np.random.seed(42)
    X = np.random.normal(size=(batch_size, seq_len, embed_dim)).astype(np.float32)
    X = tf.convert_to_tensor(X)

    # Create an instance of your SWA layer
    swa_layer = SWA(
        embed_dim=embed_dim,
        max_seq_len=seq_len,  # Set to a reasonable max sequence length
        n_heads=n_heads,
        n_kv_heads=n_kv_heads,
        window_size=window_size
    )

    # Test causal mode
    causal_output = swa_layer(X, causal=True)
    print(f"Causal output shape: {causal_output.shape}")

    # Test non-causal mode
    non_causal_output = swa_layer(X, causal=False)
    print(f"Non-causal output shape: {non_causal_output.shape}")


   # print("SWA traiable params:\n")
    #print(swa_layer.trainable_variables)

    return causal_output, non_causal_output

In [ ]:
causal, non_causal=test_sliding_window_attention()

Causal output shape: (1, 16, 64)
Non-causal output shape: (1, 16, 64)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'swa', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


# Transformer block

In [ ]:
class Transformer(tf.keras.layers.Layer):
  def __init__(self, embed_dim, max_seq_len, n_heads, n_kv_heads, window_size,multiple_of=256, custom_mult=None,  dropout=0.2):
    super().__init__()
    self.RMSNorm1=RMSNorm(embed_dim)
    self.RMSNorm2=RMSNorm(embed_dim)
    self.FFN=FFN(embed_dim, multiple_of=multiple_of, custom_mult=custom_mult, dropout=dropout)
    self.SWA=SWA(embed_dim, max_seq_len, n_heads, n_kv_heads, window_size)

  def call(self, x):
    x1=self.RMSNorm1(x)
    x2=self.SWA(x1)
    x3=x+x2
    x4=self.RMSNorm2(x3)
    x5=self.FFN(x4)
    x6=x3+x5
    return x6



In [ ]:
## Mistral
class Mistral(tf.keras.Model):
  def __init__(self, vocab_size, n_layers, embed_dim, max_seq_len, n_heads, n_kv_heads, window_size, multiple_of=256, custom_mult=None, dropout=0.2):
    super().__init__()
    self.embedding=tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
    self.transformer_layers=tf.keras.Sequential([Transformer(embed_dim, max_seq_len, n_heads, n_kv_heads, window_size, multiple_of=multiple_of, custom_mult=custom_mult, dropout=dropout) for _ in range(n_layers)])
    self.RMSNorm=RMSNorm(embed_dim)
    self.head=tf.keras.layers.Dense(vocab_size, use_bias=False)

  def call(self, x):
    h=self.embedding(x)
    x=self.transformer_layers(h)
    print(x.shape)
    x=self.RMSNorm(x)
    print(x.shape)
    x=self.head(x)
    return x

In [ ]:
#Test Mistral
x=tf.keras.random.randint(minval=0, maxval=100, shape=(1, 10))
mistral=Mistral(100, 2, 32, 64, 4, 2, 4)
mistral(x).shape


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'swa_9', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'swa_10', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


(1, 10, 32)
(1, 10, 32)
(1, 10, 32)
(1, 10, 32)


TensorShape([1, 10, 100])